FlowCharts : Analyse des sorties prématurées (Dropouts) et causes d'arrêt

In [ ]:
## On commence par importer la librairie pandas
import pandas as pd

## On charge ensuite les fichiers dm
dm01 = pd.read_csv("dm1.csv")
dm02 = pd.read_csv("dm2.csv")

print(dm01['ARM'].value_counts())
print(dm02['ARM'].value_counts())
## On utilise la fonction concat pour regrouper les 2
dm = pd.concat([dm01, dm02], ignore_index=True)

## On vérifie ici qu'on a bien des patients unique
print((dm['USUBJID'].value_counts() > 1).sum())

# Description des valeurs des patients
print(dm['ARM'].value_counts())

dmIncluded = dm[dm['ARM'] != "SCREEN FAILURE"]
print(dmIncluded['ARM'].value_counts())

# On fait ici la même chose que pour dm
ds01 = pd.read_csv("ds1.csv")
ds02 = pd.read_csv("ds2.csv")

ds = pd.concat([ds01, ds02], ignore_index=True)

dsDischarged = (ds[ds['EPOCH'] != "SCREENING"]
                .assign(beforeEndActive=lambda x: x['VISITNUM'] < 14)
                .query('beforeEndActive & DSDECOD != "PARTICIPANT COMPLETED ACTIVE PHASE OF STUDY"')
                [['STUDYID', 'USUBJID', 'DSDECOD', 'VISITNUM']]
                .drop_duplicates())

print(dsDischarged['VISITNUM'].value_counts())
print(dsDischarged['DSDECOD'].value_counts())

dtf = dmIncluded.merge(dsDischarged, on='USUBJID', how='left')

# On affiche ensuite le résultat final
crosstab = pd.crosstab(dtf['ARMCD'], dtf['VISITNUM_y'])
print(crosstab.sum(axis=1))

print(pd.crosstab(dtf['DSDECOD'], dtf['ARMCD']))

ARM
BUPRENORPHINE/NALOXONE    77
CLONIDINE                 36
SCREEN FAILURE            25
Name: count, dtype: int64
ARM
BUPRENORPHINE/NALOXONE    156
CLONIDINE                  74
SCREEN FAILURE             43
Name: count, dtype: int64
0
ARM
BUPRENORPHINE/NALOXONE    233
CLONIDINE                 110
SCREEN FAILURE             68
Name: count, dtype: int64
ARM
BUPRENORPHINE/NALOXONE    233
CLONIDINE                 110
Name: count, dtype: int64
VISITNUM
3     21
2     10
10     8
1      6
4      5
8      4
7      2
13     1
5      1
Name: count, dtype: int64
DSDECOD
PARTICIPANT WITHDREW FROM STUDY                                                                   17
VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM (INDICATE TYPE)                                13
VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM-METHADONE                                       8
ADMINISTRATIVELY WITHDRAWN                                                                         5
PARTICIPANT NO LONGER A